# EDA — Câncer de Mama (Wisconsin)

## 1. Imports e configuração

In [1]:
import copy

# Workaround: bug em matplotlib.path.Path.__deepcopy__ com Python 3.14
from matplotlib.path import Path as _MplPath
def _fixed_path_deepcopy(self, memo):
    cls = self.__class__
    p = cls.__new__(cls)
    memo[id(self)] = p
    for k, v in self.__dict__.items():
        try:
            setattr(p, k, copy.deepcopy(v, memo))
        except Exception:
            setattr(p, k, v)
    p._readonly = False
    return p
_MplPath.__deepcopy__ = _fixed_path_deepcopy

import matplotlib
matplotlib.use('Agg')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

## 2. Carregamento do dataset

In [2]:
df = pd.read_csv('data/data.csv')

# Remove coluna vazia gerada pelo CSV e o ID (não é feature)
df.drop(columns=['Unnamed: 32', 'id'], inplace=True)

print('Shape:', df.shape)
df.head()

Shape: (569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3. Tipos, missing values e consistência

In [3]:
print('=== Tipos de dados ===')
print(df.dtypes)
print()
print('=== Valores ausentes ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'Nenhum valor ausente.')

=== Tipos de dados ===
diagnosis                   object
radius_mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
smoothness_worst           float64
compactness_worst          float64
concavity_worst            float

In [4]:
print('=== Valores únicos em diagnosis ===')
print(df['diagnosis'].value_counts())
print()
print('Linhas duplicadas:', df.duplicated().sum())

=== Valores únicos em diagnosis ===
diagnosis
B    357
M    212
Name: count, dtype: int64

Linhas duplicadas: 0


## 4. Estatísticas descritivas

In [5]:
df.describe().T[['mean', 'std', 'min', 'max']].round(3)

,mean,std,min,max
radius_mean,14.127,3.524,6.981,28.110
texture_mean,19.290,4.301,9.710,39.280
perimeter_mean,91.969,24.299,43.790,188.500
area_mean,654.889,351.914,143.500,2501.000
smoothness_mean,0.096,0.014,0.053,0.163
compactness_mean,0.104,0.053,0.019,0.345
concavity_mean,0.089,0.080,0.000,0.427
concave points_mean,0.049,0.039,0.000,0.201
symmetry_mean,0.181,0.027,0.106,0.304
fractal_dimension_mean,0.063,0.007,0.050,0.097


Escalas bem diferentes entre features — normalização vai ser necessária antes da modelagem.

## 5. Distribuição da variável alvo

In [6]:
counts = df['diagnosis'].value_counts()
pct = df['diagnosis'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Benigno (B)', 'Maligno (M)'], counts.values, color=['#66c2a5', '#fc8d62'], edgecolor='white')
axes[0].set_title('Contagem por Diagnóstico')
axes[0].set_ylabel('Amostras')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Benigno (B)', 'Maligno (M)'],
            autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Proporção por Diagnóstico')

plt.suptitle('Distribuição da Variável Alvo', fontsize=13, fontweight='bold')
plt.savefig('figures/01_target_distribution.png', bbox_inches='tight')
plt.show()

print(f'Benigno: {counts["B"]} ({pct["B"]:.1f}%)  |  Maligno: {counts["M"]} ({pct["M"]:.1f}%)')

Benigno: 357 (62.7%)  |  Maligno: 212 (37.3%)


/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_27164/4125608074.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


63% benigno, 37% maligno. Dataset desbalanceado, mas não extremo. No contexto médico, o que importa é não perder os malignos, então a métrica principal vai ser recall da classe M.

## 6. Distribuição das features por diagnóstico

In [7]:
mean_features = [c for c in df.columns if c.endswith('_mean')]

fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, feat in enumerate(mean_features):
    for label, color in zip(['B', 'M'], ['#66c2a5', '#fc8d62']):
        axes[i].hist(df[df['diagnosis'] == label][feat], bins=30,
                     alpha=0.6, color=color, label='Benigno' if label == 'B' else 'Maligno', edgecolor='white')
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuição das Features (_mean) por Diagnóstico', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('figures/02_feature_histograms.png', bbox_inches='tight')
plt.show()

/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_27164/1116301872.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`radius_mean`, `area_mean` e `concave points_mean` separam bem as classes. `fractal_dimension_mean` e `texture_mean` têm distribuições quase idênticas entre B e M.

## 7. Boxplots — features _mean por diagnóstico

In [8]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, feat in enumerate(mean_features):
    sns.boxplot(data=df, x='diagnosis', y=feat, ax=axes[i],
                hue='diagnosis', palette={'B': '#66c2a5', 'M': '#fc8d62'},
                order=['B', 'M'], legend=False)
    axes[i].set_xticklabels(['Benigno', 'Maligno'])
    axes[i].set_xlabel('')
    axes[i].set_title(feat)

plt.suptitle('Boxplots das Features (_mean) por Diagnóstico', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('figures/03_feature_boxplots.png', bbox_inches='tight')
plt.show()

/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/seaborn/_base.py:1447: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.plot_data[axis] = cat_data
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/se

/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_27164/868071554.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Medianas de malignos bem acima dos benignos. Tem vários outliers nos benignos que parecem grandes mas são benignos — esses são os casos difíceis.

## 8. Matriz de correlação

In [9]:
corr = df[mean_features].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8}, ax=ax)
ax.set_title('Correlação entre Features (_mean)', fontsize=13, fontweight='bold')
plt.savefig('figures/04_correlation_matrix.png', bbox_inches='tight')
plt.show()

/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_27164/4179618145.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`radius`, `perimeter` e `area` têm correlação acima de 0.99. Geometricamente faz sentido. Vou manter as três mas considerar isso na interpretação dos coeficientes da Logística.

## 9. Pairplot — features mais discriminativas

In [10]:
top_features = ['radius_mean', 'concave points_mean', 'area_mean', 'compactness_mean', 'diagnosis']

g = sns.pairplot(df[top_features], hue='diagnosis',
                 palette={'B': '#66c2a5', 'M': '#fc8d62'},
                 plot_kws={'alpha': 0.5}, diag_kind='kde')
g.fig.suptitle('Pairplot — Features mais Discriminativas', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('figures/05_pairplot.png', bbox_inches='tight')
plt.show()

/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_27164/3600797513.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


A separação é boa — qualquer modelo razoável vai funcionar. Os casos difíceis ficam na sobreposição das nuvens de pontos.